# Long Pipe Valve — R-THYM Cross-Engine Verification

Mirrors **`tests/test_long_pipe_valve.py`** and Appendix B.1–B.5: five-pipe equal-percentage closure vs `tests/R-THYM_MOC_Verification.json` and `tests/R-THYM_MOC_Traces.csv`.

> **Runtime:** full re-simulation is ~2–3 minutes (warmup + 232 s transient). Set `RUN_SIMULATION = False` to plot reference traces only.

[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jlillywh/RTHYM-MOC/main?labpath=examples%2Flong_pipe_valve_verification.ipynb)

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rthym_moc
from _verification_notebook_setup import bootstrap_verification_notebook

REPO_ROOT, TESTS_DIR = bootstrap_verification_notebook()
print(f"Repository root: {REPO_ROOT}")
print(f"rthym_moc: {getattr(rthym_moc, '__version__', 'unknown')}")

## 2. Run (optional) and metrics

In [ ]:
from long_pipe_valve_verification_utils import (
    TRACE_PSI_TOL_RMS,
    load_reference,
    run_and_evaluate_long_pipe,
)

RUN_SIMULATION = True  # False = reference CSV only

ref, csv = load_reference()
if RUN_SIMULATION:
    results, ref, csv, metrics = run_and_evaluate_long_pipe()
    print(f"Overall PASS: {metrics.passed}")
else:
    results = None
    metrics = None
    print("Skipped simulation; plotting R-THYM reference only.")

## 3. Pressure trace overlay

In [ ]:
mask = (csv["t"] >= 35.0) & (csv["t"] <= 65.0)
t_ref = csv["t"][mask]
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_ref, csv["vp"][mask], "k-", label="R-THYM reference (Valve_B)")
if results is not None:
    from long_pipe_valve_verification_utils import _WARMUP_S, interp_to_ref
    sim_t = np.asarray(results["time"])
    sim_p = interp_to_ref(t_ref + _WARMUP_S, sim_t, np.asarray(results["node_pressure"]["Valve_B"]))
    ax.plot(t_ref, sim_p, "r--", label="RTHYM-MOC")
    rms = metrics.trace_rms_psi["Valve_B"]
    ax.set_title(f"Post-closure window — RMS err {rms:.2f} psi (tol {TRACE_PSI_TOL_RMS})")
ax.set_xlabel("R-THYM time (s)")
ax.set_ylabel("Pressure (psi)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()